# Prompt Engineering

A small language model runs on a laptop. Larger ones need hardware a laptop does not have, so they are used through an **API**: your code sends text to a provider and gets the answer back. Here you call `gpt-oss-20b`, a model with 21 billion parameters, through the API of [Groq](https://groq.com/).

What you send is the **prompt**, and its wording decides how useful the answer is. **Prompt engineering** is writing and improving prompts until the answers are reliable. One task runs through the notebook: Tretwerk, a fictional company renting e-bikes by monthly subscription, receives customer emails that its support team wants a model to sort and answer.

## Learning Objectives

By the end of this notebook, you should be able to:

- Call a language model through an API and read the answer and its token usage.
- Build a prompt from its ingredients and chain prompts with LangChain.

## Setup

In [ ]:
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

load_dotenv()  # reads GROQ_API_KEY from the .env file

> **Note.** The free Groq plan allows about 8,000 tokens per minute. If a cell fails with a rate limit error, wait a minute and run it again; the model below also retries on its own.

## 1. Calling a Model Through an API

[LangChain](https://python.langchain.com/) gives many providers the same interface. [`ChatGroq`](https://reference.langchain.com/python/langchain-groq/chat_models/ChatGroq) connects to Groq; switching provider later means changing this one line. The arguments you will use most:

| Argument | Meaning |
|---|---|
| `model` | which model the provider runs |
| `temperature` | how random the sampling is; `0` gives (almost) the same answer every time |
| `max_tokens` | the most tokens the model may generate, which also caps the cost of a call |
| `max_retries` | how often LangChain retries a failed call, for example after a rate limit |
| `reasoning_effort` | how long the model thinks before answering (section 2); `"low"` keeps calls fast and cheap |

`invoke` sends one prompt and waits for the answer.

In [ ]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=1000,
    max_retries=5,
    reasoning_effort="low",
)

reply = llm.invoke("In one sentence: what is an e-bike?")
print(reply.content)

The answer is an [`AIMessage`](https://reference.langchain.com/python/langchain-core/messages/ai/AIMessage). Besides the text in `content`, it reports the **tokens** the call used. Providers charge per token, so this is the number to track.

In [ ]:
print(reply.usage_metadata)
print(reply.response_metadata["finish_reason"])

**Result.** `input_tokens` counts the prompt, about 80 tokens for an eight-word question: the provider wraps it in the model's chat template, with its special tokens and instructions. `output_tokens` counts everything generated, and `output_token_details` shows part of it was **reasoning**, text written before the answer (section 2). `finish_reason` is `stop`: the model ended the answer itself.

### Messages and Roles

A chat model receives a list of **messages**, each with a **role**:

- **system** sets the model's role and the rules for the whole conversation;
- **human** is what the user writes;
- **ai** holds the model's earlier answers.

A chat template turns these roles into the special tokens the model was trained on; with an API the provider applies it. In LangChain you pass the messages as `(role, text)` pairs.

In [ ]:
messages = [
    (
        "system",
        "You are a support agent of Tretwerk, a company that rents e-bikes by subscription. Answer in two sentences at most.",
    ),
    ("human", "How much does the Urban plan cost per month?"),
]
print(llm.invoke(messages).content)

**Result.** The price is invented: in this run the model said 29.99 euros and added features such as unlimited rides. Tretwerk does not exist, so it cannot know the prices, but the system message made it a support agent and it answers in that tone anyway. A language model produces the most plausible text, not the true one.

## 2. Reasoning Models

Models answer reasoning questions better when asked to **think step by step**. This is **chain-of-thought** prompting, and each written-out step becomes context for the next token.

Many current models, `gpt-oss-20b` among them, are **reasoning models**, trained to produce that chain of thought on their own. Groq returns it separately from the answer, in `additional_kwargs`.

In [ ]:
question = "A customer pays 49 euros per month and pays twelve months in advance, getting two months free. How much do they pay?"

reply = llm.invoke(question)
print("ANSWER:   ", reply.content)
print("REASONING:", reply.additional_kwargs["reasoning_content"])
print(reply.usage_metadata["output_token_details"])

**Result.** The answer is 490 euros, which matches Tretwerk's annual Urban plan. The reasoning shows the steps: 12 months at 49 euros, minus the two free months. They are visible only because the provider returns them, without being asked for in the prompt.

Reasoning is paid for like any other output token. `reasoning_effort` chooses how much of it: `"low"`, `"medium"` (Groq's default) or `"high"`. The cell below asks the same question at each level, capped at 2,000 tokens, and counts the reasoning tokens.

In [ ]:
rows = []
for effort in ["low", "medium", "high"]:
    model = ChatGroq(
        model="openai/gpt-oss-20b",
        temperature=0,
        max_tokens=2000,
        max_retries=5,
        reasoning_effort=effort,
    )
    reply = model.invoke(question)
    rows.append(
        {
            "effort": effort,
            "reasoning tokens": reply.usage_metadata["output_token_details"][
                "reasoning"
            ],
            "finish reason": reply.response_metadata["finish_reason"],
            "answer": reply.content,
        }
    )

pd.DataFrame(rows)

**Result.** All three reach the same conclusion at very different cost: in this run `low` used 67 reasoning tokens and `medium` about 800. `high` spent 1,998 of its 2,000 tokens reasoning and returned an empty answer with `finish_reason` `length`, because **`max_tokens` counts the reasoning tokens too**. Higher effort helps with multi-step calculations; for short tasks `low` is enough. If an answer comes back empty, check `finish_reason` first.

## 3. The Ingredients of a Prompt

A prompt can be a single question, but reliable prompts are built from a few ingredients:

| Ingredient | What it says | Example |
|---|---|---|
| **Persona** | which role the model takes | *You are a support agent at Tretwerk.* |
| **Instruction** | the task, as precise as possible | *Summarize the customer's email.* |
| **Context** | what the model needs to know for the task | *Tretwerk rents e-bikes by subscription.* |
| **Format** | the shape of the answer | *One sentence, then a list of open questions.* |
| **Audience** | who reads the answer | *The agent replying to the customer.* |
| **Tone** | the style of the answer | *Neutral and factual.* |
| **Data** | the input the task is about | the email itself |

[`ChatPromptTemplate`](https://reference.langchain.com/python/langchain-core/prompts/chat/ChatPromptTemplate) turns a prompt with placeholders such as `{email}` into messages, so one prompt serves every email. The emails are in `data/emails.csv`.

Persona, Instruction, Format, Audience and Tone are how you word the prompt. Context and Data are what you put into it, which notebooks 02 and 03 are about.

In [ ]:
emails = pd.read_csv(Path("data") / "emails.csv")
email = emails.loc[0, "text"]
print(email)

First a prompt with the instruction and the data only:

In [ ]:
bare_prompt = ChatPromptTemplate.from_messages(
    [("human", "Summarize this email: {email}")]
)

print(llm.invoke(bare_prompt.invoke({"email": email})).content)

Now the same task with all the ingredients. Printing the filled template shows what the model receives.

In [ ]:
full_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                "You are a support agent at Tretwerk, a company that rents e-bikes by monthly subscription. "
                "You prepare incoming customer emails for the agent who will reply."
            ),
        ),
        (
            "human",
            (
                "Summarize the customer's email below in one neutral sentence. "
                "Then list what the agent needs to ask the customer, as at most three bullet points. "
                "If nothing is missing, write 'Nothing to ask.'\n\n"
                "Email: {email}"
            ),
        ),
    ]
)

print(full_prompt.invoke({"email": email}).to_string())

In [ ]:
print(llm.invoke(full_prompt.invoke({"email": email})).content)

**Result.** The bare prompt gives a correct summary, but the model decides its length and form. The full prompt produces the requested shape: one sentence, then at most three questions, written for the agent rather than the customer. Note what is missing: the model does not know that `E-07` means *motor overheated* in Tretwerk's handbook, so it asks about firmware instead. Better wording cannot fix missing knowledge; that is the context's job, in notebook 02.

## 4. Chains

So far, each cell filled the template, called the model and printed `content` by hand. The **LangChain Expression Language** (LCEL) connects these steps with `|` into a **chain**: each step's output is the next one's input. [`StrOutputParser`](https://reference.langchain.com/python/langchain-core/output_parsers/string/StrOutputParser) at the end turns the `AIMessage` into its text.

In [ ]:
summary_chain = full_prompt | llm | StrOutputParser()

summary = summary_chain.invoke({"email": email})
print(summary)

Chains also break a task into steps. This is **chain prompting**: instead of one long prompt asking for everything, each step gets a short prompt with one job whose output you can check. The second step below drafts a reply from the first step's summary. A dictionary at the start of a chain runs the chains inside it and passes their results on under these keys.

<img src="assets/chain_two_steps.png" alt="The email goes through full_prompt, llm and parser, which together form summary_chain, and comes out as a summary. The summary goes through reply_prompt, llm and a parser and comes out as the reply. The whole sequence is reply_chain." width="1000">

In [ ]:
reply_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a friendly support agent at Tretwerk. You never promise refunds or dates.",
        ),
        (
            "human",
            "Write a short reply to the customer, based on this summary of their email:\n\n{summary}",
        ),
    ]
)

reply_chain = {"summary": summary_chain} | reply_prompt | llm | StrOutputParser()

print(reply_chain.invoke({"email": email}))

**Result.** The reply is friendly and asks for missing details, as the summary suggested. It also asks for the firmware version, which Tretwerk would never need, because no step in the chain knows the handbook either. Each step can be checked on its own: if the reply is wrong, you can see whether the summary already was.

**Your turn:** add a third step to the chain that translates the reply into German, or into another language you read. Write the prompt, then build `translated_chain` from `reply_chain`, the prompt, the model and a parser, and run it on email 1.

In [ ]:
# TODO: a prompt with a {reply} placeholder that asks for a translation
translate_prompt = ...

translated_chain = ...
print(translated_chain.invoke({"email": emails.loc[1, "text"]}))

## Summary

- Large models are used through an **API**. `ChatGroq` returns an `AIMessage` with the answer in `content` and token counts in `usage_metadata`. Tokens are what providers charge and limit.
- A chat model receives **messages** with roles: **system** for the role and rules, **human** from the user, **ai** for earlier answers.
- **Reasoning models** such as `gpt-oss-20b` write a chain of thought before answering; `reasoning_effort` sets how long, and its tokens are paid for like the answer. Models that do not can be asked to think step by step instead.
- Reliable prompts combine **persona, instruction, context, format, audience, tone and data**.
- **LCEL** chains connect a prompt, a model and a parser with `|`, and chains can feed each other, so a task splits into small steps.
- For answers with several fields, providers can also return **structured output** following a schema, such as JSON.

## Sources

- Wei, J. et al. (2022). [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903).
- [Groq documentation: reasoning](https://console.groq.com/docs/reasoning) describes `reasoning_effort` and the reasoning output.
- Tretwerk and its emails are fictional, written for this repository.